In [5]:
# Cell 1 - paths + Kely's working notebook (installs MapAnything/SAM3 env exactly as before)
import json, os, glob
CAP = '/content/drive/MyDrive/Capstone-w-HERE'
EXP_ROOT = '/content/drive/MyDrive/capstone_clearance_nbi_expanded'
ORIG_ROOT = '/content/drive/MyDrive/capstone_clearance_nbi'
NB_PATH = f'{CAP}/Weekly Update Artifacts/new quarter- kely/kely_mapanything_nbi_expanded.ipynb'
V2 = f'{CAP}/pipeline_v2'; os.makedirs(V2, exist_ok=True)
nb = json.load(open(NB_PATH))
SRC = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
for i, s in enumerate(SRC): print(i, len(s), s.strip().split(chr(10))[1][:90] if chr(10) in s.strip() else s[:90])
print(os.listdir(EXP_ROOT)[:20]); print(os.path.exists(ORIG_ROOT) and os.listdir(ORIG_ROOT)[:20])
get_ipython().run_cell(SRC[0])

0 1293 # 0. Install SAM3 + MapAnything (run once per runtime; auto-restarts the session after a f
1 4919 # 1. Config, Drive output folder, tokens, Mapillary helpers
2 519 import shutil; NEW_ROOT = Path("/content/drive/MyDrive/capstone_clearance_nbi_expanded"); 
3 3464 # 2. Ground truth: FHWA National Bridge Inventory (Illinois)
4 6322 # 3. For each NBI bridge, find a Mapillary sequence driving UNDER it (heading toward it) a
5 1413 # 4. Load models
6 9271 # 5. Core functions: Mask2Former (SAM3 replacement) selection -> MapAnything 3D -> road-pl
7 8582 # 6. Per-case processing (resumable: results saved per case on Drive)
8 924 # 7. RUN all cases (safe to re-run after a disconnect: finished cases are loaded from Driv
9 4042 # 8. Results vs NBI ground truth
10 571 # 9. Contact sheet of all anchor overlays (for visual QA)
11 4033 # 10. Second object type: STOP signs (MUTCD R1-1) -> sign width vs standard size
12 6521 # 11. Stop-sign width from MapAnything: width_m = sign_width_px * depth / 

<ExecutionResult object at 7cae68eb12e0, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7cae68eb1440, raw_cell="# =========================
# 0. Install SAM3 + Ma.." store_history=False silent=False shell_futures=True cell_id=None> result=None>

In [6]:
# Cell 2 - load Kely's pipeline functions without running their loops; logs go to pipeline_v2/logs
import ast, os, json, sys, io, contextlib, traceback
os.makedirs(f'{V2}/logs', exist_ok=True)
def logged(name, fn):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        try: fn()
        except Exception: traceback.print_exc()
    open(f'{V2}/logs/{name}.txt', 'w').write(buf.getvalue()); print(buf.getvalue()[-2500:])
def _exec_defs(src, skip_loops=True):
    tree = ast.parse('\n'.join(l for l in src.split('\n') if not l.lstrip().startswith(('!', '%'))))
    keep = [n for n in tree.body if not (skip_loops and isinstance(n, (ast.For, ast.While)))]
    exec(compile(ast.Module(body=keep, type_ignores=[]), '<kely>', 'exec'), globals())
def load_all():
    for i in [1, 3, 4, 5, 6, 7]:
        print('loading cell', i, flush=True)
        try: _exec_defs(SRC[i])
        except Exception: print('ERROR in cell', i); traceback.print_exc()
    import inspect
    for fn in ['plan_case', 'download_case', 'process_case']:
        if fn in globals(): print(fn, inspect.signature(globals()[fn]))
    print([k for k in globals() if k.isupper()])
    bc = json.load(open(f'{ORIG_ROOT}/cases.json')); print('benchmark cases:', len(bc), list(bc[0].keys()))
    for i in [3, 4]: print('==== SRC', i); print(SRC[i][:3500])
logged('c2_load', load_all)
from pathlib import Path
def use_run(name):
    global OUT_ROOT, IMG_DIR, CASE_DIR, VIS_DIR
    OUT_ROOT = Path(V2) / 'runs' / name
    IMG_DIR, CASE_DIR, VIS_DIR = OUT_ROOT / 'images', OUT_ROOT / 'case_results', OUT_ROOT / 'visualizations'
    for d in (IMG_DIR, CASE_DIR, VIS_DIR): d.mkdir(parents=True, exist_ok=True)
    return OUT_ROOT

,case_id,gt_clearance_m,service_on,structure_kind,feature_under,facility_on,sequence,n_views,anchor_d_m
0,nbi_016261000000000,4.65,highway,prestressed concrete continuous,I-294 TRI STATE TOLL,55TH ST,zhmfdjsy0r3l17u0cco3a8,8,45.738973
1,nbi_099018800000000,5.00,3rd-level interchange,steel continuous,I- 80,CENTER ST NB,YKqE8Hshfol6ewBdU4jOnk,8,66.946353
2,nbi_016256100000000,5.00,highway,steel continuous,IL 390 ELGIN-OHARE,ROSELLE RD,z6v7icp3iqorehbenpgkc6,8,46.136771
3,nbi_099033300000000,4.98,interchange overpass,prestressed concrete continuous,I-355,IL 171,kcn9oaoeado0nl0djy4weg,7,38.642613
4,nbi_016106200000000,5.49,3rd-level interchange,steel continuous,ARCHER BNSF & IC RR,I- 90 94 SB TOI-55,k68w504togsazovxxyutq6,7,43.563545
...,...,...,...,...,...,...,...,...,...
75,nbi_016170700000000,5.18,interchange overpass,steel continuous,I- 90 94 JFK,VAN BUREN ST,d046zy6oczm76ox95cc9b7,8,44.855076
76,nbi_016978100000000,4.34,highway,prestressed concrete,LAWRENCE AVE,I-294 SB,n70x9c3rz3jrqc9rjw75wg,8,43.733168
77,nbi_032010000000000,4.34,interchange overpass,steel continuous,FAS285/CH5/RIDGE RD,FAI 80 (EB & WB),p1uy2SMIRmJZNFevWaC6zl,4,68.018583
78,nbi_016072400000000,7.42,highway,steel continuous,I- 55 STEVENSON,CENTRAL AVE & RAMP,5RpYHCPlz64twS2ouKbrJq,8,45.246610


preprocessor_config.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/79.5k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  866MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/758 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.91GB            

model.safetensors: downloading bytes:           |  0.00B            

": 2000})
    rows = data.get("data", [])
    if not rows:
        return None, "no_images"
    df = pd.DataFrame(rows)
    geo = df.get("computed_geometry", pd.Series([None] * len(df))).where(lambda s: s.notna(), df["geometry"])
    df["lon"], df["lat"] = zip(*geo.apply(_xy))
    head = df.get("computed_compass_angle", pd.Series([np.nan] * len(df)))
    df["heading"] = pd.to_numeric(head, errors="coerce").fillna(pd.to_numeric(df.get("compass_angle"), errors="coerce"))
    df = df.dropna(subset=["lon", "lat", "heading", "thumb_2048_url", "sequence"])
    if "camera_type" in df:
        df = df[~df["camera_type"].fillna("perspective").isin(["spherical", "equirectangular", "fisheye"])]
    if df.empty:
        return None, "no_perspective_images"
    df["d_m"] = [haversine_m(x, y, b.lon, b.lat) for x, y in zip(df.lon, df.lat)]
    df["bearing"] = [bearing_deg(x, y, b.lon, b.lat) for x, y in zip(df.lon, df.lat)]
    df["head_err"] = [ang_diff(h, br) for h, br in zip(df.heading, df.bearing

In [8]:
# Cell 3 - inspect the per-case API
def show():
    print(SRC[4][3400:]); print('#'*60); print(SRC[7]); print('#'*60); print(SRC[1][1800:4919])
logged('c3_src', show)

g multi-view run
SEARCH_RADIUS_M = 170
MIN_D_M, MAX_D_M = 12, 140        # camera-to-bridge GPS distance for usable frames
ANCHOR_D_RANGE = (22, 85)
ANCHOR_TARGET_D_M = 45
HEADING_TOL_DEG = 25
STRUCT_PROMPTS = ["bridge", "overpass", "highway overpass", "bridge underside"]
ROAD_PROMPTS = ["road", "road surface", "highway"]
MIN_STRUCT_SCORE, MIN_ROAD_SCORE = 0.30, 0.30
FORCE_RERUN = False
RESULT_VERSION = "v2_above_cam_filter"   # bump to recompute cached case results
SEED = 7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2FT = 3.28084
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("OUT_ROOT:", OUT_ROOT)
print("numpy", np.__version__, "| torch", torch.__version__)

# ---- Mapillary + geo helpers ----
def mapillary_get(url, params=None, retries=4):
    params = dict(params or {})
    params["access_token"] = MAPILLARY_TOKEN
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=60)
    

In [10]:
bench_cases = json.load(open(f'{ORIG_ROOT}/cases.json'))
# Exp A - map-matched crossing retrieval for the 42 benchmark bridges that produced no anchor
import requests, math, time, json, numpy as np, pandas as pd
from shapely.geometry import LineString
OVERPASS = ['https://overpass-api.de/api/interpreter', 'https://overpass.kumi.systems/api/interpreter']
def overpass_ways(lat, lon, radius=150):
    q = f"""[out:json][timeout:60];
    (way(around:{radius},{lat},{lon})["highway"]["bridge"];
     way(around:{radius},{lat},{lon})["highway"][!"bridge"][!"tunnel"];);
    out tags geom;"""
    for attempt in range(4):
        url = OVERPASS[attempt % 2]
        try:
            r = requests.post(url, data={'data': q}, timeout=90)
            if r.status_code == 200: return r.json()['elements']
        except Exception: pass
        time.sleep(4 * (attempt + 1))
    raise RuntimeError('overpass failed')

def find_crossing(lat, lon, feature_under='', radius=150):
    els = overpass_ways(lat, lon, radius)
    line = lambda e: LineString([(p['lon'], p['lat']) for p in e['geometry']])
    bridges = [e for e in els if e.get('tags', {}).get('bridge') not in (None, 'no') and len(e.get('geometry', [])) > 1]
    roads = [e for e in els if 'bridge' not in e.get('tags', {}) and len(e.get('geometry', [])) > 1
             and e['tags'].get('highway') not in ('footway', 'cycleway', 'path', 'steps', 'service', 'track', 'pedestrian', 'corridor')]
    fu = ''.join(ch for ch in str(feature_under).upper() if ch.isalnum())
    best = None
    for b in bridges:
        bl = line(b)
        for rd in roads:
            rl = line(rd); inter = bl.intersection(rl)
            if inter.is_empty: continue
            pt = inter if inter.geom_type == 'Point' else inter.centroid
            d = haversine_m(lon, lat, pt.x, pt.y)
            name = ''.join(ch for ch in (rd['tags'].get('ref', '') + rd['tags'].get('name', '')).upper() if ch.isalnum())
            match = bool(fu) and (fu[:5] in name or name[:5] in fu) if name else False
            score = d - (60 if match else 0)
            if best is None or score < best[0]:
                proj = rl.project(pt); a = rl.interpolate(max(proj - 1e-4, 0)); c = rl.interpolate(proj + 1e-4)
                best = (score, dict(lat=pt.y, lon=pt.x, under_bearing=bearing_deg(a.x, a.y, c.x, c.y),
                                    under_ref=rd['tags'].get('ref') or rd['tags'].get('name', ''), offset_m=d, name_match=match,
                                    n_bridges=len(bridges)))
    return best[1] if best else None

_orig_mget = mapillary_get
UNDER_BRG = None
def mapillary_get_filtered(url, params=None, retries=4):
    data = _orig_mget(url, params, retries)
    if UNDER_BRG is None or not isinstance(data, dict) or 'data' not in data: return data
    keep = []
    for r in data['data']:
        h = r.get('computed_compass_angle', r.get('compass_angle'))
        if h is not None and min(ang_diff(h, UNDER_BRG), ang_diff(h, UNDER_BRG + 180)) <= 30: keep.append(r)
    data['data'] = keep
    return data

def expA():
    global UNDER_BRG
    use_run('A_crossing')
    orig = {c['case_id']: c for c in bench_cases}
    rows = []
    for cid, c in orig.items():
        o = json.load(open(f'{ORIG_ROOT}/case_results/{cid}.json'))
        if o.get('status') not in ('no_structure_on_anchor', 'no_road_on_anchor'): continue
        row = dict(case_id=cid, gt=c['gt_clearance_m'], orig_status=o.get('status'))
        try:
            cx = find_crossing(c['lat'], c['lon'], c.get('feature_under', ''))
        except Exception as e:
            cx = None; row['err'] = f'overpass:{e}'
        row['crossing_found'] = cx is not None
        if cx:
            row.update(offset_m=cx['offset_m'], under_ref=cx['under_ref'], name_match=cx['name_match'])
            b = pd.Series({**c, 'lat': cx['lat'], 'lon': cx['lon']})
            UNDER_BRG = cx['under_bearing']; globals()['mapillary_get'] = mapillary_get_filtered
            try: plan, why = plan_case(b)
            except Exception as e: plan, why = None, f'error:{e}'
            globals()['mapillary_get'] = _orig_mget; UNDER_BRG = None
            row['plan'] = why
            if plan:
                case = {**c, **plan, 'crossing_lat': cx['lat'], 'crossing_lon': cx['lon']}
                try:
                    case = download_case(case); rec = process_case(case)
                    row.update({k: rec.get(k) for k in ['status', 'qc_pass', 'anchor_year', 'single_clear_med_m', 'multi_clear_med_m',
                                                          'multi_allframes_med_m', 'single_cam_height_m', 'multi_cam_height_m',
                                                          'multi_n_cols', 'multi_struct_dist_m', 'anchor_image_id', 'anchor_gps_d_m']})
                except Exception as e:
                    row['err'] = f'process:{e}'
        rows.append(row); print(row, flush=True)
        pd.DataFrame(rows).to_csv(f'{V2}/expA_crossing.csv', index=False)
    df = pd.DataFrame(rows)
    print('crossing found', df.crossing_found.sum(), '/', len(df))
    if 'status' in df: print(df.status.value_counts(dropna=False)); print('qc_pass', df.qc_pass.sum())
logged('expA', expA)

KeyboardInterrupt: 

In [ ]:
# Exp B - lens undistortion with Mapillary camera parameters (same frames, same code, only the pixels change)
import cv2, shutil
REC_KEYS = ['status', 'qc_pass', 'anchor_year', 'anchor_image_id', 'anchor_gps_d_m', 'single_clear_med_m', 'single_clear_p10_m',
            'multi_clear_med_m', 'multi_clear_p10_m', 'multi_allframes_med_m', 'single_cam_height_m', 'multi_cam_height_m',
            'single_n_cols', 'multi_n_cols', 'single_struct_dist_m', 'multi_struct_dist_m', 'single_reason', 'multi_reason']
_cam_cache = {}
def cam_info(image_id):
    if image_id not in _cam_cache:
        try: _cam_cache[image_id] = mapillary_get(f'https://graph.mapillary.com/{image_id}', {'fields': 'camera_parameters,camera_type,make,model,captured_at'})
        except Exception as e: _cam_cache[image_id] = {'err': str(e)}
    return _cam_cache[image_id]

def undistort_file(src, dst, params):
    img = cv2.imread(src); h, w = img.shape[:2]
    if not params or len(params) < 3: shutil.copy(src, dst); return False
    f, k1, k2 = params[:3]; fpx = f * max(w, h)
    K = np.array([[fpx, 0, w / 2], [0, fpx, h / 2], [0, 0, 1]], float); dist = np.array([k1, k2, 0, 0], float)
    if abs(k1) < 1e-4 and abs(k2) < 1e-4: shutil.copy(src, dst); return False
    newK, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 0)
    cv2.imwrite(dst, cv2.undistort(img, K, dist, None, newK), [cv2.IMWRITE_JPEG_QUALITY, 95]); return True

def expB():
    use_run('B_undistort')
    rows = []
    for c in bench_cases:
        cid = c['case_id']
        o = json.load(open(f'{ORIG_ROOT}/case_results/{cid}.json'))
        row = dict(case_id=cid, gt=c['gt_clearance_m'], orig_status=o.get('status'), orig_qc=o.get('qc_pass'),
                   orig_single=o.get('single_clear_med_m'), orig_multi=o.get('multi_clear_med_m'), orig_allframes=o.get('multi_allframes_med_m'))
        a_id = o.get('anchor_image_id') or c['frames'][c['anchor_index']]['image_id']
        ci = cam_info(a_id); p = ci.get('camera_parameters')
        row.update(make=ci.get('make'), model=ci.get('model'), camera_type=ci.get('camera_type'),
                   focal=p[0] if p else None, k1=p[1] if p and len(p) > 1 else None, k2=p[2] if p and len(p) > 2 else None)
        if o.get('status') in ('no_structure_on_anchor', 'no_road_on_anchor'):
            rows.append(row); continue
        cdir = IMG_DIR / cid; cdir.mkdir(exist_ok=True); case = json.loads(json.dumps(c)); changed = 0
        for k, fr in enumerate(case['frames']):
            src = fr.get('local_path') or str(Path(ORIG_ROOT) / 'images' / cid / f"{k:02d}_{fr['image_id']}.jpg")
            src = src.replace('/capstone_clearance_nbi_expanded/', '/capstone_clearance_nbi/')
            dst = str(cdir / Path(src).name)
            changed += undistort_file(src, dst, cam_info(fr['image_id']).get('camera_parameters')); fr['local_path'] = dst
        row['frames_undistorted'] = changed
        try:
            rec = process_case(case); row.update({k: rec.get(k) for k in REC_KEYS})
        except Exception as e: row['err'] = str(e)
        rows.append(row); print({k: row.get(k) for k in ['case_id', 'gt', 'make', 'k1', 'orig_single', 'single_clear_med_m', 'orig_multi', 'multi_clear_med_m', 'qc_pass']}, flush=True)
        pd.DataFrame(rows).to_csv(f'{V2}/expB_undistort.csv', index=False)
    pd.DataFrame(rows).to_csv(f'{V2}/expB_undistort.csv', index=False)
logged('expB', expB)

# Exp D - external validation: same frozen pipeline in two new metros
METROS = {'IN': dict(lat_min=39.60, lat_max=40.00, lon_min=-86.40, lon_max=-85.90),   # Indianapolis
          'WI': dict(lat_min=42.85, lat_max=43.25, lon_min=-88.20, lon_max=-87.80)}   # Milwaukee
def expD(st, n=60):
    use_run(f'D_{st}')
    globals()['REGION_BBOX'] = METROS[st]; globals()['N_TARGET_CASES'] = n
    exec(SRC[3].replace('IL_nbi.txt', f'{st}_nbi.txt').replace('delimited/IL{', 'delimited/' + st + '{'), globals())
    exec(SRC[4], globals())
    rows = []
    for case in globals()['cases']:
        try: rec = process_case(case)
        except Exception as e: rec = {'status': f'error:{e}'}
        row = dict(case_id=case['case_id'], metro=st, gt=case['gt_clearance_m'], service_on=case.get('service_on'),
                   lat=case['lat'], lon=case['lon'], **{k: rec.get(k) for k in REC_KEYS})
        if rec.get('anchor_image_id'):
            ci = cam_info(rec['anchor_image_id']); p = ci.get('camera_parameters')
            row.update(make=ci.get('make'), model=ci.get('model'), k1=p[1] if p and len(p) > 1 else None)
        rows.append(row); print(row['case_id'], row['gt'], row['status'], row.get('multi_clear_med_m'), flush=True)
        pd.DataFrame(rows).to_csv(f'{V2}/expD_{st}.csv', index=False)
for st in METROS: logged(f'expD_{st}', lambda st=st: expD(st))

In [ ]:
# overpass connectivity test
def t():
    import requests
    q = '[out:json][timeout:25];way(around:150,41.97657,-87.86969)["highway"]["bridge"];out tags;'
    for url in OVERPASS + ['https://maps.mail.ru/osm/tools/overpass/api/interpreter', 'https://overpass.private.coffee/api/interpreter']:
        for hdr in [{}, {'User-Agent': 'uchicago-capstone/1.0 (kelyistudying@gmail.com)'}]:
            try:
                r = requests.post(url, data={'data': q}, headers=hdr, timeout=60)
                print(url, bool(hdr), r.status_code, r.text[:150].replace(chr(10), ' '))
            except Exception as e: print(url, bool(hdr), 'EXC', e)
logged('overpass_test', t)